In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "hepach2020help")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "cpat_data_f.txt")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_table(complete_path_1)


In [3]:
df['study_id']="hepach2020help"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "ape", 
    "stooge":"ape_2"}, inplace=True)


In [4]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')

In [5]:
df = df.assign(role='focal_participant')
df = df.assign(role_2='stooge')


In [6]:
# df[['day','month', 'year']] = df['date'].str.split('/',expand=True)
# df['year'] = '20' + df['year'].astype(str)


datedf=[]
for index, row in df.iterrows():
    if "/" in str(row['date']):
        day,month,year = str(row['date']).split('/')
        if year == '18':
            year = '2018'
        if year == '19':
            year = '2019'
        datedf.append([day,month,year])
    else:
        # print(str(row['date']).split('.'))except
        try:
            day,month,year = str(row['date']).split('.')
            datedf.append([day,month,year])
        except:
            # print(str(row['date']))
            datedf.append(["","",""])

df[["day","month",  "year"]] = datedf

In [7]:
df.rename(columns={"ape":"participant", 
    "ape_2":"participant_2"}, inplace=True)

In [8]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [9]:
df.rename(columns={"chimp_dyad_stoo_sub":"chimp_dyad_stooge_focal",
                   'group':'subgroup',
                   'stooge_subject_together':'stooge_focal_participant_together',
                   'position_sub_reach_stooge':'position_focal_reach_stooge',
                   'subject_gives':'focal_participant_gives',
                   'reach_stooge_when_subject_push':'reach_stooge_when_focal_participant_push',
                   'time_response_subject':'time_response_focal_participant',
                   'session_day_subject':'session_day_focal_participant'}, inplace=True)

In [10]:
# df['paternalism'].replace('strong', 'non-occluded', inplace=True, regex=True)
# df['paternalism'].replace('weak', 'occluded', inplace=True, regex=True)

In [11]:
trial_do_list = [[1, "stooge_did_not_reach"],
                 [2,"stooge_reached_but_subject_did_not_approach_panel"],
                 [3,"stooge_got_tool_themself"],
                 [4,"equipment_failure"],
                 [5,"experimenter_error"],
                 [10,"session_aborted"]]

for x,y in trial_do_list:
    df.loc[df.trial_do_crit == x, ['trial_do_code_explained']] = y

df.rename(columns={"trial_do_crit":"trial_do_code",
                   'stooge_focal_participant_together':'session', 
                   'condition':'condition_remove'}, inplace=True)

In [12]:
df.rename(columns={'subgroup':'species_subgroup',
                   'session_nr_type':'condition'}, inplace=True)
df['year'].replace('2018', '2014', inplace=True, regex=True)
df['year'].replace('2019', '2015', inplace=True, regex=True)

In [13]:
helping_list = ['weak_test', 'strong_test']
spe_2=[]  
for index, row in df.iterrows():
    if row['condition'] in helping_list:
        if row['stooge_reaches_for_when_nail remove'] == "reach_for_irr":
            spe_2.append("paternalism")
        elif row['stooge_reaches_for_when_nail remove'] == "reach_for_rel":
            spe_2.append("helping")
        else:
            spe_2.append("")
    else:
        spe_2.append("")
df = df.assign(new_column=spe_2)

spe_3=[]  
for index, row in df.iterrows():
    if row['condition'] in helping_list:
        spe_3.append("need")
    else:
        spe_3.append("no_need")
df = df.assign(condition_type=spe_3)


df['paternalism'].replace('weak', 'yes_weak', inplace=True, regex=True)
df['paternalism'].replace('strong', 'no_strong', inplace=True, regex=True)
df['paternalism'].unique()



array(['yes_weak', 'no_strong'], dtype=object)

In [14]:

df.rename(columns={'new_column':'help_option',
                   "paternalism":"tool_occlusion"}, inplace=True)

In [15]:

hepach2020help_standardized=df[['study_id',  'year', 'month','day',
                                 'participant','age_in_years','sex',  'role',
                                   'participant_2','age_in_years_2','sex_2', 'role_2', 'species','dyad', 
                                   'chimp_dyad_stooge_focal','session',
        'trial','trial_drop', 'trial_do_code','trial_do_code_explained','species_subgroup',
         'test_session_date_sequence', 
         'session_day_stooge', 
        'session_day_focal_participant', 
       'condition','condition_type', 'tool_occlusion', 'help_option',
        'stooges_first_reach',
       'stooge_reaches_for_when_nail remove', 'position_focal_reach_stooge',
        'focal_participant_gives', 'reach_stooge_when_focal_participant_push',
       
       'latency_sec', 'session_abort', 'drop_criterion' ]]
comp_out_path_stand = os.path.join(out_pathway, 'hepach2020help_exp2_standardized.csv')
hepach2020help_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =hepach2020help_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
hepach2020help_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'hepach2020help_exp2_glossary.csv')
hepach2020help_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
